# Text Classification with Classical Machine Learning

## 📚 Learning Objectives

By completing this notebook, you will:
- Turn raw text into numeric features with **TF-IDF**
- Train **two** text classifiers — Naive Bayes and Logistic Regression — and **compare** them
- Evaluate with a held-out test set **and** cross-validation
- See, measured live, **why tiny datasets give unreliable evaluations**

## 🔗 Where this fits

**Builds on:** Course 04 (AIAT 114) — Unit 3, lessons 01 and 05 (logistic regression, Naive Bayes) and Unit 2, lesson 01 (cross-validation) — the same classifiers and the same evaluation, now on TF-IDF text features.

**Used later in:** Course 07 — Unit 5, whose bias audit inspects classifiers built this way.

---


## 📥 Inputs & 📤 Outputs

**Inputs:** 32 short movie reviews written into this notebook (16 positive, 16 negative) — no downloads.

**Outputs:** a TF-IDF feature matrix, accuracy and per-class metrics for two classifiers, cross-validation
scores, and a measured demonstration of small-dataset instability.

---

## Part 1: From Text to Numbers — TF-IDF

Classifiers need numeric inputs, so each document becomes a vector of word scores. **TF-IDF** (Term
Frequency × Inverse Document Frequency) scores a word highly in a document when it appears **often in that
document** (TF) but **rarely across the other documents** (IDF). Words that appear everywhere — and generic
stop words like "the" — get little or no weight, so the surviving features are the words that actually
distinguish documents. This is the workhorse text representation for classical ML.


In [1]:
# Build a small labeled dataset and turn the text into numbers with TF-IDF.
# Classifiers cannot read words — TF-IDF gives each review a numeric vector that
# scores a word highly when it is frequent HERE but rare across ALL reviews.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# A small labeled dataset: 16 positive and 16 negative movie reviews
positive_reviews = [
    "This movie is amazing and wonderful",
    "I love this film, it's fantastic and fun",
    "Great acting and a wonderful story",
    "Excellent film with great cinematography",
    "A wonderful, fun and heartwarming movie",
    "Brilliant, amazing performances by the cast",
    "I love this movie, highly recommended",
    "One of the best films of the year, amazing",
    "Beautiful story, excellent acting, great fun",
    "Fantastic soundtrack and a great story",
    "A brilliant film full of wonderful moments",
    "Amazing direction and excellent acting, loved it",
    "Fun, fantastic and beautifully made",
    "The best movie this year, wonderful acting",
    "Loved the story, great film, highly recommended",
    "The plot is slow at times, but the acting is wonderful and the ending is amazing",
]
negative_reviews = [
    "Terrible movie, very boring",
    "Worst film I have ever seen, awful",
    "Poor quality, boring and not recommended",
    "Boring and slow from start to finish",
    "Awful acting and a terrible plot",
    "A complete waste of time, very poor",
    "Disappointing film with terrible dialogue",
    "Dull story, flat characters, boring",
    "Bad editing and a confusing, awful plot",
    "I regret watching this boring mess",
    "Cheap effects, horrible pacing, terrible",
    "Painfully slow, dull and forgettable",
    "The worst acting I have seen, awful film",
    "Terrible pacing and a dull, bad story",
    "Poor script, boring scenes, disappointing",
    "A great cast completely wasted on a boring, terrible script",
]
documents = positive_reviews + negative_reviews
labels = ["positive"] * len(positive_reviews) + ["negative"] * len(negative_reviews)
print(f"Dataset: {len(documents)} reviews "
      f"({labels.count('positive')} positive / {labels.count('negative')} negative)")

# TF-IDF feature extraction
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(documents)
features = vectorizer.get_feature_names_out()

print(f"\nTF-IDF matrix: {X.shape[0]} documents x {X.shape[1]} features (one per distinct word)")
print(f"Sample features: {sorted(features)[:10]} ...")

# Peek at one document's strongest features
doc0 = X[0].toarray().ravel()
top = np.argsort(doc0)[::-1][:4]
print(f"\nDocument 1: {documents[0]!r}")
print("Its highest TF-IDF features:", [(str(features[i]), round(float(doc0[i]), 3)) for i in top])

Dataset: 32 reviews (16 positive / 16 negative)

TF-IDF matrix: 32 documents x 64 features (one per distinct word)
Sample features: ['acting', 'amazing', 'awful', 'bad', 'beautiful', 'beautifully', 'best', 'boring', 'brilliant', 'cast'] ...

Document 1: 'This movie is amazing and wonderful'
Its highest TF-IDF features: [('amazing', 0.588), ('movie', 0.588), ('wonderful', 0.555), ('year', 0.0)]


## Part 2: Train and Compare Two Classifiers

Quick refreshers:

- **Train/test split**: we hold out 25% of documents the models never see during training, so the accuracy
  we report measures *generalization*, not memorization. `stratify` keeps the class balance in both halves.
- **Multinomial Naive Bayes**: learns how likely each word is inside each class, then multiplies those
  likelihoods (assuming words are independent — "naive", but famously effective on text counts).
- **Logistic Regression**: learns one weight per word — positive weights push toward one class, negative
  toward the other — and squashes the weighted sum into a probability.

We evaluate both with the same held-out test set, and then with **8-fold cross-validation** (train/evaluate
8 times on different splits and average), which uses the small dataset more efficiently than a single split.


In [2]:
# Train and compare two classic text classifiers on the SAME features.
# Naive Bayes and Logistic Regression are the standard first models for text:
# fast, strong baselines you should always try before anything deep.

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25, random_state=42, stratify=labels
)
print(f"Training on {X_train.shape[0]} reviews, testing on {X_test.shape[0]} held-out reviews\n")

# Two candidate models trained on identical TF-IDF features — a fair head-to-head.
classifiers = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
}

results = {}
# Fit on the training split, score on held-out data, and cross-validate for stability.
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    predictions = clf.predict(X_test)
    holdout_acc = accuracy_score(y_test, predictions)
    cv_scores = cross_val_score(clf, X, labels, cv=8)
    results[name] = (holdout_acc, cv_scores.mean(), cv_scores.std(), predictions)

    print("=" * 64)
    print(name)
    print("=" * 64)
    for true, pred in zip(y_test, predictions):
        mark = "✓" if true == pred else "✗"
        print(f"  true: {true:8s}  predicted: {pred:8s}  {mark}")
    print(f"  Held-out accuracy:        {holdout_acc:.1%}")
    print(f"  8-fold CV accuracy:       {cv_scores.mean():.1%} (+/- {cv_scores.std():.1%})")
    print()

# Detailed per-class metrics for the first classifier
nb_pred = results["Multinomial Naive Bayes"][3]
print("Per-class metrics (Naive Bayes, held-out test set):")
print(classification_report(y_test, nb_pred))

# Computed comparison
(nb_h, nb_cv, _, _) = results["Multinomial Naive Bayes"]
(lr_h, lr_cv, _, _) = results["Logistic Regression"]
if abs(nb_cv - lr_cv) < 1e-9:
    print(f"Comparison: both classifiers score identically here (CV {nb_cv:.1%}).")
elif nb_cv > lr_cv:
    print(f"Comparison: Naive Bayes wins on CV ({nb_cv:.1%} vs {lr_cv:.1%}).")
else:
    print(f"Comparison: Logistic Regression wins on CV ({lr_cv:.1%} vs {nb_cv:.1%}).")
print("On real, larger datasets the two often differ more; on this small clean dataset")
print("read the comparison from the numbers above, not from folklore.")

Training on 24 reviews, testing on 8 held-out reviews

Multinomial Naive Bayes
  true: positive  predicted: positive  ✓
  true: positive  predicted: positive  ✓
  true: negative  predicted: negative  ✓
  true: negative  predicted: negative  ✓
  true: positive  predicted: positive  ✓
  true: positive  predicted: positive  ✓
  true: negative  predicted: negative  ✓
  true: negative  predicted: negative  ✓
  Held-out accuracy:        100.0%
  8-fold CV accuracy:       100.0% (+/- 0.0%)

Logistic Regression
  true: positive  predicted: positive  ✓
  true: positive  predicted: positive  ✓
  true: negative  predicted: negative  ✓
  true: negative  predicted: negative  ✓
  true: positive  predicted: positive  ✓
  true: positive  predicted: positive  ✓
  true: negative  predicted: negative  ✓
  true: negative  predicted: negative  ✓
  Held-out accuracy:        100.0%
  8-fold CV accuracy:       100.0% (+/- 0.0%)

Per-class metrics (Naive Bayes, held-out test set):
              precision    re

## Part 3: Why Are These Scores So High? (And When Not to Trust a Score)

Look at the accuracies you just computed. They are high because this toy dataset is **easy**: positive
reviews reuse words like *amazing/wonderful/great*, negative ones reuse *boring/terrible/awful*, so TF-IDF
features separate the classes almost perfectly. Real-world text — sarcasm, mixed opinions, typos, topic
drift — is far messier, and scores like these should make you suspicious, not proud.

There is a second trap: **evaluation on tiny datasets is unstable**. To *measure* that instead of just
claiming it, the next cell shrinks the dataset to 8 reviews (2 test documents) and re-runs the identical
pipeline with five different random splits. Watch the "accuracy" swing.


In [3]:
# Stress test: rerun the same pipeline with only 8 documents to expose a trap.
# Tiny test sets make accuracy a lottery — this is the most common way beginners
# fool themselves about model quality.

tiny_docs = positive_reviews[:4] + negative_reviews[:4]
tiny_labels = ["positive"] * 4 + ["negative"] * 4

print("Same pipeline, but only 8 documents (2 in the test set), 5 different splits:")
tiny_accs = []
# Same model, five different random train/test splits — watch the score jump around.
for seed in range(5):
    v = TfidfVectorizer(stop_words="english")
    Xt = v.fit_transform(tiny_docs)
    Xtr, Xte, ytr, yte = train_test_split(Xt, tiny_labels, test_size=0.25, random_state=seed)
    acc = accuracy_score(yte, MultinomialNB().fit(Xtr, ytr).predict(Xte))
    tiny_accs.append(acc)
    print(f"  split seed {seed}: accuracy = {acc:.0%}")

print(f"\nAccuracy across splits: min {min(tiny_accs):.0%}, max {max(tiny_accs):.0%}, "
      f"mean {np.mean(tiny_accs):.0%}")
print("Same data, same model — the score depends mostly on WHICH 2 documents landed in")
print("the test set. With so few test samples, a single accuracy number is noise.")
print("Lessons: (1) never trust an evaluation with a handful of test samples;")
print("(2) prefer cross-validation on small data; (3) get more data when you can.")

Same pipeline, but only 8 documents (2 in the test set), 5 different splits:
  split seed 0: accuracy = 100%
  split seed 1: accuracy = 100%
  split seed 2: accuracy = 100%
  split seed 3: accuracy = 0%
  split seed 4: accuracy = 0%

Accuracy across splits: min 0%, max 100%, mean 60%
Same data, same model — the score depends mostly on WHICH 2 documents landed in
the test set. With so few test samples, a single accuracy number is noise.
Lessons: (1) never trust an evaluation with a handful of test samples;
(2) prefer cross-validation on small data; (3) get more data when you can.


---

## ✅ Summary

**What you actually did here** (all numbers computed live above):

- Built TF-IDF features from 32 labeled reviews and inspected which words score highly
- Trained and **compared** Multinomial Naive Bayes and Logistic Regression, with a held-out test set,
  per-class precision/recall/F1, and 8-fold cross-validation
- Measured evaluation instability on a tiny 8-document dataset: the same pipeline's accuracy swung between
  the minimum and maximum you printed, purely from the random split

**Honest limitations:** the 32-review dataset is deliberately easy and vocabulary-separable, so the scores
here are near the ceiling; real sentiment data (you will meet some in the exercise) is much harder.

**Next:** `02_named_entity_recognition.ipynb` — extracting structured information with spaCy.


## 📚 References

Where these ideas come from — seminal work first, then a modern survey:

1. Salton, G., & Buckley, C. (1988). *Term-Weighting Approaches in Automatic Text Retrieval*. Information Processing & Management, 24(5), 513-523.
2. Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed. draft) — chapter on Naive Bayes, text classification, and sentiment. <https://web.stanford.edu/~jurafsky/slp3/>
3. Minaee, S., Kalchbrenner, N., Cambria, E., et al. (2021). *Deep Learning-Based Text Classification: A Comprehensive Review*. ACM Computing Surveys. [arXiv:2004.03705](https://arxiv.org/abs/2004.03705)
